### Inspección y limpieza de los datos

En este seminario trabajaremos con el **Spanish Fake News Corpus**, un conjunto de noticias en español etiquetadas como **verdaderas** o **falsas** (bulos). Fue creado por investigadores de la UNAM y el IPN (México) y se ha usado en las competiciones MEX-A3T e IberLEF ([repositorio original](https://github.com/jpposadas/FakeNewsCorpusSpanish), licencia CC BY 4.0).

El corpus tiene dos partes recogidas en épocas distintas, y eso lo vamos a aprovechar:

- **Noticias de 2018** (`train.xlsx` y `development.xlsx`): las usaremos para entrenar y validar nuestros modelos.
- **Noticias de 2020-2021** (`test.xlsx`): las guardaremos como **"noticias nuevas"**, publicadas años después y con temas que no existían en 2018 (¡hola, COVID-19!). Nos servirán para comprobar si los modelos generalizan.

Este notebook está estructurado como sigue:

- **Importación de librerías**
- **BLOQUE A**: carga de los datos.
- **BLOQUE B**: inspección de los datos.
- **BLOQUE C**: limpieza y adecuación.
- **BLOQUE D**: exportación de los conjuntos resultantes.

In [1]:
# Preparación del entorno: funciona tanto en Google Colab como en local
import os, sys

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB and not os.path.exists("seminario-manadine-transfer-learning"):
    !git clone -q https://github.com/Vins16/seminario-manadine-transfer-learning.git
    %pip install -q -r seminario-manadine-transfer-learning/requirements.txt

if EN_COLAB:
    RAIZ = "seminario-manadine-transfer-learning"
else:
    RAIZ = "." if os.path.exists("data") else ".."
sys.path.insert(0, RAIZ)
DATA = f"{RAIZ}/data"
MODELS = f"{RAIZ}/models"

### Importación de librerías

In [2]:
import pandas as pd
import re

### BLOQUE A: carga de los datos

Los datos vienen en ficheros Excel, así que usaremos la función de pandas que lee este formato.

In [3]:
df_train = pd.read_excel(f"{DATA}/raw/train.xlsx")
df_dev = pd.read_excel(f"{DATA}/raw/development.xlsx")
df_nuevas = pd.read_excel(f"{DATA}/raw/test.xlsx")

### BLOQUE B: inspección de los datos

El objetivo de la inspección es familiarizarnos con el conjunto de datos. Algunas preguntas que conviene hacerse:

- ¿En qué tipo de objeto están almacenados los datos?
- ¿Cuál es su dimensión?
- ¿Hay datos ausentes?
- ¿Las distintas partes del corpus tienen el mismo formato?

In [4]:
# Tipo de objeto con el que estamos trabajando
type(df_train)

pandas.DataFrame

In [5]:
# Dimensiones de cada conjunto
print(f"Entrenamiento 2018: {df_train.shape[0]} filas y {df_train.shape[1]} columnas")
print(f"Desarrollo 2018:    {df_dev.shape[0]} filas y {df_dev.shape[1]} columnas")
print(f"Nuevas 2020-21:     {df_nuevas.shape[0]} filas y {df_nuevas.shape[1]} columnas")

Entrenamiento 2018: 676 filas y 7 columnas
Desarrollo 2018:    295 filas y 7 columnas
Nuevas 2020-21:     572 filas y 7 columnas


In [6]:
# Primeras filas del conjunto de entrenamiento
df_train.head()

,Id,Category,Topic,Source,Headline,Text,Link
0,1,Fake,Education,El Ruinaversal,"RAE INCLUIRÁ LA PALABRA ""LADY"" EN EL DICCIONAR...","RAE INCLUIRÁ LA PALABRA ""LADY"" EN EL DICCIONAR...",http://www.elruinaversal.com/2017/06/10/rae-in...
1,2,Fake,Education,Hay noticia,"La palabra ""haiga"", aceptada por la RAE","La palabra ""haiga"", aceptada por la RAE La Rea...",https://haynoticia.es/la-palabra-haiga-aceptad...
2,3,Fake,Education,El Ruinaversal,YORDI ROSADO ESCRIBIRÁ Y DISEÑARÁ LOS NUEVOS L...,YORDI ROSADO ESCRIBIRÁ Y DISEÑARÁ LOS NUEVOS L...,http://www.elruinaversal.com/2018/05/06/yordi-...
3,4,True,Education,EL UNIVERSAL,UNAM capacitará a maestros para aprobar prueba...,UNAM capacitará a maestros para aprobar prueba...,http://www.eluniversal.com.mx/articulo/nacion/...
4,5,Fake,Education,Lamula,pretenden aprobar libros escolares con conteni...,Alerta: pretenden aprobar libros escolares con...,https://redaccion.lamula.pe/2018/06/19/memoria...


In [7]:
# Primeras filas de las noticias nuevas. ¿Qué diferencias ves respecto al anterior?
df_nuevas.head()

,ID,CATEGORY,TOPICS,SOURCE,HEADLINE,TEXT,LINK
0,1,True,Covid-19,El Economista,Covid-19: mentiras que matan,El control de la Covid-19 no es sólo un tema d...,https://www.eleconomista.com.mx/opinion/Covid-...
1,2,False,Política,El matinal,El Gobierno podrá acceder a las IPs de los móv...,El Gobierno de Pedro Sánchez y Pablo Iglesias ...,https://www.elmatinal.com/espana-ultima-hora/e...
2,3,True,Política,El País,La comunidad musulmana catalana denuncia a Vox...,Las tres federaciones que agrupan al 90% de la...,https://elpais.com/espana/elecciones-catalanas...
3,4,False,Política,AFPFactual,NaN,Se han dado a conocer los datos electorales pr...,https://perma.cc/GYE6-SPMB
4,5,True,Sociedad,La Republica,El censo poblacional 2018 tendrá un costo de $...,La primera fase del censo será virtual y solo ...,https://www.larepublica.co/economia/el-censo-p...


Hay varias diferencias de formato que tendremos que arreglar:

1. Los nombres de las columnas están en **MAYÚSCULAS** en las noticias nuevas.
2. La etiqueta está escrita como `"Fake"`/`"True"` en 2018 y como booleano (`True`/`False`) en 2020-21.
3. Los temas están en inglés en 2018 y en español en 2020-21.

In [8]:
# Resumen conciso de cada DataFrame: tipos de datos y valores no nulos
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 676 entries, 0 to 675
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Id        676 non-null    int64
 1   Category  676 non-null    str  
 2   Topic     676 non-null    str  
 3   Source    676 non-null    str  
 4   Headline  676 non-null    str  
 5   Text      676 non-null    str  
 6   Link      676 non-null    str  
dtypes: int64(1), str(6)
memory usage: 1.7 MB


In [9]:
df_nuevas.info()

<class 'pandas.DataFrame'>
RangeIndex: 572 entries, 0 to 571
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   ID        572 non-null    int64
 1   CATEGORY  572 non-null    bool 
 2   TOPICS    572 non-null    str  
 3   SOURCE    565 non-null    str  
 4   HEADLINE  500 non-null    str  
 5   TEXT      572 non-null    str  
 6   LINK      569 non-null    str  
dtypes: bool(1), int64(1), str(5)
memory usage: 1.9 MB


In [10]:
# ¿Cuántos valores ausentes hay en cada columna de las noticias nuevas?
df_nuevas.isna().sum()

ID           0
CATEGORY     0
TOPICS       0
SOURCE       7
HEADLINE    72
TEXT         0
LINK         3
dtype: int64

In [11]:
# ¿Está equilibrada la variable objetivo?
df_train["Category"].value_counts()

Category
Fake    338
True    338
Name: count, dtype: int64

In [12]:
df_nuevas["CATEGORY"].value_counts()

CATEGORY
True     286
False    286
Name: count, dtype: int64

In [13]:
# ¿De qué temas hablan las noticias de cada época?
print(df_train["Topic"].value_counts(), "\n")
print(df_nuevas["TOPICS"].value_counts())

Topic
Politics         226
Entertainment    103
Society           93
Sport             86
Science           62
Health            32
Economy           30
Security          29
Education         15
Name: count, dtype: int64 

TOPICS
Covid-19         237
Sociedad         195
Política         107
Internacional     14
Ciencia           13
Ambiental          4
Deporte            2
Name: count, dtype: int64


#### Cuidado con la fuga de información (*data leakage*)

Veamos qué relación hay entre la **fuente** de la noticia y su etiqueta:

In [14]:
# Tabla de contingencia entre la fuente y la categoría
pd.crosstab(df_train["Source"], df_train["Category"]).sort_values("Fake", ascending=False).head(10)

Category,Fake,True
Source,,
El Dizque,99,0
El Ruinaversal,55,0
Hay noticia,31,0
Censura 0,25,0
La voz popular,16,0
Retroceso,12,0
Modo Noticia,11,0
El mundo today,6,0
Modo noticia,5,0


Algunas fuentes (webs satíricas o de bulos) publican **solo** noticias falsas. Un modelo que usara la columna `Source` acertaría casi siempre en estos datos, pero no aprendería nada útil: ante una fuente desconocida no sabría qué hacer. Por eso **trabajaremos únicamente con el texto** (titular + cuerpo de la noticia).

### BLOQUE C: limpieza y adecuación

In [15]:
# Unimos los dos ficheros de 2018 en un único DataFrame
df_2018 = pd.concat([df_train, df_dev], ignore_index=True)
df_2018.shape

(971, 7)

In [16]:
# Homogeneizamos los nombres de las columnas de las noticias nuevas (por ejemplo, "CATEGORY" pasa a ser "Category")
df_nuevas.columns = df_nuevas.columns.str.capitalize()

# La columna de temas se llama distinto: cambiamos "Topics" por "Topic"
df_nuevas = df_nuevas.rename(columns={"Topics": "Topic"})
df_nuevas.columns

Index(['Id', 'Category', 'Topic', 'Source', 'Headline', 'Text', 'Link'], dtype='str')

Creamos la variable objetivo `target`, que valdrá **1 si la noticia es falsa** y **0 si es verdadera**.

En las noticias nuevas, `Category == True` significa que la noticia es *verdadera*, así que hay que tener cuidado con el orden.

In [17]:
df_2018["target"] = df_2018["Category"].map({"Fake": 1, "True": 0})
df_nuevas["target"] = df_nuevas["Category"].map({False: 1, True: 0})

print(df_2018["target"].value_counts(), "\n")
print(df_nuevas["target"].value_counts())

target
0    491
1    480
Name: count, dtype: int64 

target
0    286
1    286
Name: count, dtype: int64


In [18]:
# Traducimos los temas de 2018 al español para que ambos conjuntos sean coherentes
traduccion_temas = {
    "Politics": "Política", "Entertainment": "Entretenimiento", "Society": "Sociedad",
    "Sport": "Deporte", "Science": "Ciencia", "Health": "Salud",
    "Economy": "Economía", "Security": "Seguridad", "Education": "Educación",
}
df_2018["Topic"] = df_2018["Topic"].map(traduccion_temas)
df_2018["Topic"].value_counts()

Topic
Política           323
Entretenimiento    148
Sociedad           134
Deporte            124
Ciencia             89
Salud               46
Economía            43
Seguridad           42
Educación           22
Name: count, dtype: int64

In [19]:
# Algunas noticias nuevas no tienen titular: rellenamos los vacíos con una cadena vacía
df_nuevas["Headline"] = df_nuevas["Headline"].fillna("")
df_nuevas["Headline"].isna().sum()

np.int64(0)

Ahora limpiamos el texto. La limpieza será **ligera** a propósito: los modelos de transfer learning que usaremos en el notebook 4 se entrenaron con texto natural (con mayúsculas, tildes y signos de puntuación), así que les conviene recibirlo tal cual. La normalización más agresiva la haremos en el notebook 3, solo para el modelo clásico.

In [20]:
def limpiar_texto(texto):
    texto = str(texto)
    texto = re.sub(r"http\S+", " ", texto)   # quitamos las URLs
    texto = re.sub(r"\s+", " ", texto)       # saltos de línea y espacios repetidos se convierten en un solo espacio
    return texto.strip()


limpiar_texto("  Comparte   esto:\n\nhttps://bulo.com/noticia  ¡YA!  ")

'Comparte esto: ¡YA!'

In [21]:
for df in [df_2018, df_nuevas]:
    df["Headline"] = df["Headline"].apply(limpiar_texto)
    df["Text"] = df["Text"].apply(limpiar_texto)

#### Un detalle que se escapa fácilmente: los números

Fíjate en este fragmento de una noticia de 2018:

In [22]:
df_2018["Text"].iloc[0][900:1100]

'n su idioma original ("dama" en inglés) sino que se usará para definir a una mujer que es problemática o acostumbra causar problemas y alborotos. La gente podrá decirle Lady a una fémina que cause alg'

In [23]:
# Proporción de noticias que contienen "*NUMBER*" y proporción que contienen algún dígito, en cada época
for nombre, df in [("2018", df_2018), ("2020-21", df_nuevas)]:
    con_number = df["Text"].str.contains("*NUMBER*", regex=False).mean()
    con_digitos = df["Text"].str.contains(r"\d").mean()
    print(f"{nombre:<8} *NUMBER*: {con_number:.0%}   dígitos: {con_digitos:.0%}")

2018     *NUMBER*: 77%   dígitos: 5%
2020-21  *NUMBER*: 0%   dígitos: 92%


Los autores del corpus de 2018 sustituyeron cada número por la marca `*NUMBER*`, pero en las noticias de 2020-21 los números aparecen tal cual. Si no lo corregimos, el modelo podría aprender que "contener `*NUMBER*`" es una pista sobre la clase de la noticia, y esa pista no existiría nunca en las noticias nuevas. Aplicamos el mismo tratamiento a ambos conjuntos:

In [24]:
def anonimizar_numeros(texto):
    return re.sub(r"\b\d+(?:[.,]\d+)*\b", "*NUMBER*", texto)


print(anonimizar_numeros("Murieron 1.250 personas en 2020"))

df_nuevas["Headline"] = df_nuevas["Headline"].apply(anonimizar_numeros)
df_nuevas["Text"] = df_nuevas["Text"].apply(anonimizar_numeros)

Murieron *NUMBER* personas en *NUMBER*


Muchas noticias repiten el titular al principio del cuerpo. Creamos una única columna `texto` que contenga el titular seguido del cuerpo, evitando duplicarlo.

In [25]:
def unir_titular(fila):
    titular, cuerpo = fila["Headline"], fila["Text"]
    if titular == "" or cuerpo.startswith(titular):
        return cuerpo
    return f"{titular}. {cuerpo}"


df_2018["texto"] = df_2018.apply(unir_titular, axis=1)
df_nuevas["texto"] = df_nuevas.apply(unir_titular, axis=1)
df_2018["texto"].iloc[0][:300]

'RAE INCLUIRÁ LA PALABRA "LADY" EN EL DICCIONARIO DEL IDIOMA ESPAÑOL COMO DEFINICIÓN DE "MUJER PROBLEMÁTICA" España.- El presidente de la Real Academia Española (RAE), Darío Villanueva, informó en conferencia de prensa que a partir del próximo mes se incluirá el término "Lady" como una nueva palabra '

In [26]:
# ¿Hay noticias duplicadas?
print(df_2018["texto"].duplicated().sum(), df_nuevas["texto"].duplicated().sum())

1 0


In [27]:
# Eliminamos los duplicados para que una misma noticia no acabe a la vez en entrenamiento y validación
df_2018 = df_2018.drop_duplicates(subset="texto", ignore_index=True)
df_2018.shape

(970, 9)

In [28]:
# Longitud de cada noticia en caracteres
for df in [df_2018, df_nuevas]:
    df["longitud"] = df["texto"].str.len()

df_2018["longitud"].describe()

count      970.000000
mean      2355.453608
std       1722.925758
min        173.000000
25%       1354.250000
50%       1894.000000
75%       2813.250000
max      15342.000000
Name: longitud, dtype: float64

In [29]:
# ¿Tienen la misma longitud las noticias falsas (1) y las verdaderas (0)?
df_2018.groupby("target")["longitud"].median()

target
0    2341.0
1    1620.0
Name: longitud, dtype: float64

### BLOQUE D: exportación de los conjuntos resultantes

Guardamos solo las columnas que nos serán útiles en los siguientes notebooks.

In [30]:
columnas = ["Id", "Topic", "texto", "longitud", "target"]

df_2018[columnas].to_csv(f"{DATA}/noticias_2018.csv", index=False)
df_nuevas[columnas].to_csv(f"{DATA}/noticias_2020_21.csv", index=False)